# Create FINEP FUNTTEL Awards

Creates OpenAlex award rows from Finep's official FUNTTEL contracted-project workbooks.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/finep_funttel_to_s3.py` to download the current Finep/FUNTTEL XLSX files, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** Finep's FUNTTEL dashboard at `https://premio.finep.gov.br/a-finep-externo/fontes-de-recurso/outras-fontes/o-que-e-funttel?layout=dashboard&view=dashboard`, which links to `https://download.finep.gov.br/Contratacao_Funttel.xlsx`.  
**S3 location:** `s3a://openalex-ingest/awards/finep_funttel/finep_funttel_projects.parquet`  
**Local full-source validation on 2026-05-27:** 23 official FUNTTEL contracted-project rows from Finep's current `Contratacao_Funttel.xlsx`; 100% native contract ID/title/recipient/CNPJ/municipality/state/amount/currency/signed-date/end-date/status coverage; year range 2010-2026; total BRL 315,972,787.13; total paid BRL 258,505,416.70.

**OpenAlex funder mapping:**
- funder_id: 4320322904
- display_name: `Financiadora de Estudos e Projetos`
- ROR: `https://ror.org/030w99567`
- DOI: `10.13039/501100004809`

**Mapping summary:** One OpenAlex award row per Finep contract number. `amount` and `currency` come directly from `Valor Finep` and BRL. `funder_scheme` uses the source `Demanda` when present, falling back to `Produto` / `FUNTTEL`; source recipient organization is represented as `lead_investigator.affiliation` with CNPJ retained as a source assertion.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.finep_funttel_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/finep_funttel/finep_funttel_projects.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-27 found 23 rows.
SELECT COUNT(*) as total_finep_funttel_projects
FROM openalex.awards.finep_funttel_raw;


In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.finep_funttel_raw;


In [ ]:
%sql
-- Sample raw FINEP FUNTTEL data.
SELECT
    funder_award_id,
    display_name,
    recipient_name,
    recipient_cnpj,
    demand,
    status,
    amount,
    currency,
    signed_date,
    end_date,
    landing_page_url
FROM openalex.awards.finep_funttel_raw
LIMIT 10;


## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5. Uses information_schema, not DESCRIBE as a subquery.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'finep_funttel_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|valor|currency';


In [ ]:
%sql
-- Confirm amount/date coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_with_amount,
    ROUND(MIN(TRY_CAST(amount AS DOUBLE)), 0) AS min_amount,
    ROUND(MAX(TRY_CAST(amount AS DOUBLE)), 0) AS max_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 2) AS total_amount_brl,
    ROUND(SUM(TRY_CAST(amount_paid AS DOUBLE)), 2) AS total_paid_brl,
    COUNT(signed_date) AS rows_with_signed_date,
    COUNT(end_date) AS rows_with_end_date,
    MIN(TRY_CAST(source_year AS INT)) AS min_year,
    MAX(TRY_CAST(source_year AS INT)) AS max_year
FROM openalex.awards.finep_funttel_raw;


In [ ]:
%sql
-- Native-key inspection.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT contract_number) AS distinct_contract_numbers,
    COUNT(DISTINCT ref) AS distinct_refs
FROM openalex.awards.finep_funttel_raw;


In [ ]:
%sql
-- Demand and status distribution.
SELECT
    demand,
    status,
    COUNT(*) AS cnt,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 2) AS total_brl
FROM openalex.awards.finep_funttel_raw
GROUP BY demand, status
ORDER BY cnt DESC, total_brl DESC;


In [ ]:
%sql
-- Coverage by major source fields.
SELECT
    COUNT(*) AS total,
    COUNT(description) AS has_description,
    COUNT(recipient_name) AS has_recipient_name,
    COUNT(recipient_cnpj) AS has_recipient_cnpj,
    COUNT(executor_name) AS has_executor_name,
    COUNT(municipality) AS has_municipality,
    COUNT(state) AS has_state,
    COUNT(source_workbook_url) AS has_workbook_url
FROM openalex.awards.finep_funttel_raw;


## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
-- Must return exactly 1 row. If this is empty, STOP: the funder is missing from openalex.common.funder.
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320322904;


## Step 2: Transform to OpenAlex Award Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.finep_funttel_awards
USING delta
AS
WITH
finep_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320322904
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        CASE WHEN TRY_CAST(amount AS DOUBLE) IS NOT NULL THEN 'BRL' ELSE NULL END AS parsed_currency,
        TRY_TO_DATE(signed_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(source_year AS INT) AS parsed_start_year
    FROM openalex.awards.finep_funttel_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,
        TRIM(g.display_name) as display_name,
        NULLIF(TRIM(g.description), '') as description,
        f.funder_id,
        g.native_award_id as funder_award_id,
        g.parsed_amount as amount,
        g.parsed_currency as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,
        COALESCE(NULLIF(TRIM(g.funding_type), ''), 'research') as funding_type,
        COALESCE(NULLIF(TRIM(g.demand), ''), NULLIF(TRIM(g.product), ''), 'FUNTTEL') as funder_scheme,
        'finep_funttel' as provenance,
        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_start_year) as start_year,
        COALESCE(YEAR(g.parsed_end_date), g.parsed_start_year) as end_year,
        struct(
            CAST(NULL AS STRING) as given_name,
            CAST(NULL AS STRING) as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.recipient_name), '') as name,
                'BR' as country,
                CASE
                    WHEN g.recipient_cnpj IS NOT NULL AND TRIM(g.recipient_cnpj) != '' THEN
                        ARRAY(struct(TRIM(g.recipient_cnpj) as id, 'cnpj' as type, 'source' as asserted_by))
                    ELSE CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                END as ids
            ) as affiliation
        ) as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,
        CONCAT('https://api.openalex.org/works?filter=awards.id:G',
               CAST(abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 AS STRING)) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM raw_prepared g
    CROSS JOIN finep_funder f
)
SELECT *
FROM awards_transformed;


## Step 3: Delete Previous Source Rows and Insert Priority 154

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'finep_funttel' AND priority = 154;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    154 as priority
FROM openalex.awards.finep_funttel_awards;


## Step 6: Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_finep_funttel_awards
FROM openalex.awards.finep_funttel_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.finep_funttel_awards;


In [ ]:
%sql
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_date,
    end_date,
    lead_investigator.affiliation.name AS recipient,
    landing_page_url
FROM openalex.awards.finep_funttel_awards
LIMIT 10;


In [ ]:
%sql
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT id) AS distinct_openalex_ids, COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.finep_funttel_awards;


In [ ]:
%sql
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.finep_funttel_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;


In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(landing_page_url) as has_url,
    COUNT(lead_investigator.affiliation.name) as has_recipient,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description,
    ROUND(SUM(amount), 2) as total_funding_brl
FROM openalex.awards.finep_funttel_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS rows_with_positive_amount,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount,
    ROUND(SUM(amount), 2) AS total_amount
FROM openalex.awards.finep_funttel_awards;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 2) as total_brl
FROM openalex.awards.finep_funttel_awards
GROUP BY start_year
ORDER BY start_year DESC;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 2) as total_brl
FROM openalex.awards.finep_funttel_awards
GROUP BY funder_scheme
ORDER BY cnt DESC;


In [ ]:
%sql
SELECT priority, provenance, COUNT(*) as cnt, COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids, ROUND(SUM(amount), 2) as total_brl
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'finep_funttel' AND priority = 154
GROUP BY priority, provenance;


## Admin Handoff Notes

Contractor has no S3/Databricks access. Admin must upload the parquet with `scripts/local/finep_funttel_to_s3.py`, run this notebook on Databricks, inspect the Step 6 verification cells, then run the combined CreateAwards notebook. Do not add job YAML until the Databricks ingest is run and QA is complete.
